In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\srija\OneDrive\Desktop\machine learning\DEEP LEARNING\datasets\DateFruit_Dataset.csv")

In [2]:
df.isnull().sum()
df.info()
df.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 898 entries, 0 to 897
Data columns (total 35 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   AREA           898 non-null    int64  
 1   PERIMETER      898 non-null    float64
 2   MAJOR_AXIS     898 non-null    float64
 3   MINOR_AXIS     898 non-null    float64
 4   ECCENTRICITY   898 non-null    float64
 5   EQDIASQ        898 non-null    float64
 6   SOLIDITY       898 non-null    float64
 7   CONVEX_AREA    898 non-null    int64  
 8   EXTENT         898 non-null    float64
 9   ASPECT_RATIO   898 non-null    float64
 10  ROUNDNESS      898 non-null    float64
 11  COMPACTNESS    898 non-null    float64
 12  SHAPEFACTOR_1  898 non-null    float64
 13  SHAPEFACTOR_2  898 non-null    float64
 14  SHAPEFACTOR_3  898 non-null    float64
 15  SHAPEFACTOR_4  898 non-null    float64
 16  MeanRR         898 non-null    float64
 17  MeanRG         898 non-null    float64
 18  MeanRB    

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [3]:
X = df.drop(columns=["Class"], axis=1)
y = df["Class"] 

In [4]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [5]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
scaler = StandardScaler()

X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)

### Deep Learning

In [8]:
# ANN
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [9]:
X_train_tensor = torch.tensor(X_train_scale, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scale, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)
# target values (y) must be in long format because we do cross-entropy-loss for loss function, and it expects long format.

In [10]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [12]:
# Build our model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model  = nn.Sequential(

            nn.Linear(X.shape[1], 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, 7)
        )
    
    def forward(self, x):
        return self.model(x)

In [13]:
model = ANN()

#loss and optim
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [14]:
# Training the NN

epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() # parameters update

        running_loss += loss

    train_loss = running_loss / len(train_loader)
    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 1.7137187719345093
epoch = 2/100, loss = 1.1617441177368164
epoch = 3/100, loss = 0.7665619254112244
epoch = 4/100, loss = 0.5654857158660889
epoch = 5/100, loss = 0.44595348834991455
epoch = 6/100, loss = 0.39060264825820923
epoch = 7/100, loss = 0.34487253427505493
epoch = 8/100, loss = 0.3142647445201874
epoch = 9/100, loss = 0.2969016134738922
epoch = 10/100, loss = 0.27321958541870117
epoch = 11/100, loss = 0.26819324493408203
epoch = 12/100, loss = 0.24049696326255798
epoch = 13/100, loss = 0.23089392483234406
epoch = 14/100, loss = 0.22420385479927063
epoch = 15/100, loss = 0.22176145017147064
epoch = 16/100, loss = 0.2037086933851242
epoch = 17/100, loss = 0.19386354088783264
epoch = 18/100, loss = 0.18743601441383362
epoch = 19/100, loss = 0.18092535436153412
epoch = 20/100, loss = 0.1742611974477768
epoch = 21/100, loss = 0.17400752007961273
epoch = 22/100, loss = 0.16745606064796448
epoch = 23/100, loss = 0.15979690849781036
epoch = 24/100, loss = 0.162

In [15]:
# Evaluation, and fixing steps
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1) # <- (max_value, max_value_index), _, means dont save the value

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # return actual samples in each batch.

print("total values: ", total)
print("correct values: ", correct)
print("Accuracy: ", correct/total * 100)

total values:  180
correct values:  173
Accuracy:  96.11111111111111
